In [ ]:
!pip install timm torch torchvision

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
import timm
import numpy as np
from ptflops import get_model_complexity_info

## --- Configuration ---

In [ ]:
DATA_DIR = "./Assignments_2_Datasets" 
NUM_CLASSES = 30
BATCH_SIZE = 32
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## --- Data Loading ---

In [ ]:
# Standard ImageNet transforms for pre-trained models
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## --- Dataset Splitting Trick ---

In [ ]:
# 1. Create two base datasets pointing to the SAME folder, but with different transforms
base_train_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform_train)
base_val_dataset = datasets.ImageFolder(root=DATA_DIR, transform=transform_val)

In [ ]:
# 2. Get the targets (class labels) for stratification
targets = base_train_dataset.targets
indices = np.arange(len(targets))

In [ ]:
# 3. Perform a stratified split (e.g., 80% train, 20% validation)
train_indices, val_indices = train_test_split(
    indices,
    test_size=0.20,      
    random_state=SEED, 
    stratify=targets     
)

In [ ]:
# 4. Create the final Subsets
train_dataset = Subset(base_train_dataset, train_indices)
val_dataset = Subset(base_val_dataset, val_indices)

In [ ]:
# --- DataLoaders ---
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Total images: {len(base_train_dataset)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

## --- Model Selection & Efficiency Metrics ---

In [ ]:
# Choose 3 models: ResNet50, EfficientNet-B0, ConvNeXt-Tiny
def initialize_model(model_name="resnet50"):
    model = timm.create_model(model_name, pretrained=True, num_classes=NUM_CLASSES)
    model = model.to(DEVICE)
    return model

In [ ]:
def report_efficiency(model):
    """Reports parameters, MACs, and FLOPs."""
    macs, params = get_model_complexity_info(model, (3, 224, 224), as_strings=True, print_per_layer_stat=False)
    print(f"Parameters: {params}, MACs: {macs}")

## Experiment 1

In [ ]:
def setup_linear_probe(model):
    # 1. Freeze all backbone parameters
    for param in model.parameters():
        param.requires_grad = False
        
    # 2. Unfreeze the final classification layer (timm uses 'get_classifier' or 'head')
    # timm's reset_classifier automatically replaces the head and leaves it unfrozen
    model.reset_classifier(NUM_CLASSES)
    
    # 3. Setup optimizer to ONLY train the linear classifier
    optimizer = torch.optim.Adam(model.get_classifier().parameters(), lr=1e-3)
    return model, optimizer

## Experiment 2

In [ ]:
def setup_selective_unfreezing(model, unfreeze_ratio=0.20):
    total_params = sum(p.numel() for p in model.parameters())
    target_unfrozen = total_params * unfreeze_ratio
    
    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False
        
    unfrozen_count = 0
    # Iterate backwards through parameters to unfreeze the deepest layers first
    for name, param in reversed(list(model.named_parameters())):
        if unfrozen_count < target_unfrozen:
            param.requires_grad = True
            unfrozen_count += param.numel()
        else:
            break
            
    print(f"Unfrozen {unfrozen_count/total_params * 100:.2f}% of parameters.")
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
    return model, optimizer

## Experiment 3

In [ ]:
def get_few_shot_dataloaders(dataset, percentage, seed=42):
    dataset_size = len(dataset)
    indices = list(range(dataset_size))
    split = int(np.floor(percentage * dataset_size))
    
    np.random.seed(seed)
    np.random.shuffle(indices)
    
    subset_indices = indices[:split]
    subset = Subset(dataset, subset_indices)
    
    return DataLoader(subset, batch_size=BATCH_SIZE, shuffle=True)

# train_loader_20 = get_few_shot_dataloaders(train_dataset, 0.20)


## Experiment 4

In [ ]:
from torchvision.transforms import v2

def get_corrupted_val_loader(corruption_type, severity=None):
    base_transforms = [transforms.Resize((224, 224)), transforms.ToTensor()]
    
    if corruption_type == "gaussian":
        # Pixel-level Gaussian noise (sigma=0.05, 0.1, 0.2) 
        base_transforms.append(v2.GaussianNoise(sigma=severity))
    elif corruption_type == "blur":
        # Motion blur [cite: 85]
        base_transforms.append(transforms.GaussianBlur(kernel_size=(5, 9), sigma=(0.1, 5.)))
    elif corruption_type == "brightness":
        # Brightness shift [cite: 86]
        base_transforms.append(transforms.ColorJitter(brightness=severity))
        
    base_transforms.append(transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]))
    
    transform_corrupt = transforms.Compose(base_transforms)
    corrupted_dataset = datasets.ImageFolder(root=f"{DATA_DIR}/val", transform=transform_corrupt)
    return DataLoader(corrupted_dataset, batch_size=BATCH_SIZE, shuffle=False)

## Experiment 5

In [ ]:
# Dictionary to store intermediate features
intermediate_features = {}

def get_features(name):
    def hook(model, input, output):
        # Flatten spatial dimensions (e.g., GAP) if it's a convolutional feature map
        if len(output.shape) == 4:
            output = torch.nn.functional.adaptive_avg_pool2d(output, (1, 1)).flatten(1)
        intermediate_features[name] = output.detach()
    return hook

def attach_hooks(model, model_name):

    if model_name == "resnet50":
        model.layer1.register_forward_hook(get_features('early'))
        model.layer3.register_forward_hook(get_features('middle'))
        model.layer4.register_forward_hook(get_features('final'))
    


## Training

In [ ]:
import time
import torch
import torch.nn as nn

def train_model(model, train_loader, val_loader, optimizer, num_epochs=30):
    """
    Standard training loop. 
    Use num_epochs=30 for full data, num_epochs=20 for few-shot scenarios.
    """
    criterion = nn.CrossEntropyLoss()
    
    # Dictionaries to store metrics for plotting later
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': []
    }
    
    start_time = time.time()
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0
        
        # --- Training Phase ---
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()
            
        epoch_train_loss = running_loss / total_train
        epoch_train_acc = correct_train / total_train
        
        # --- Validation Phase ---
        model.eval()
        running_val_loss = 0.0
        correct_val = 0
        total_val = 0
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                running_val_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()
                
        epoch_val_loss = running_val_loss / total_val
        epoch_val_acc = correct_val / total_val
        
        # Save metrics
        history['train_loss'].append(epoch_train_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_loss'].append(epoch_val_loss)
        history['val_acc'].append(epoch_val_acc)
        
        print(f"Epoch [{epoch+1}/{num_epochs}] "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.4f} | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.4f}")
              
    elapsed_time = time.time() - start_time
    print(f"Training completed in {elapsed_time // 60:.0f}m {elapsed_time % 60:.0f}s")
    
    return model, history

In [ ]:
# Configuration
MODELS_TO_TEST = ["resnet50"] 

def run_linear_probe(model_name):
    print(f"\n{'='*50}\nStarting Scenario 4.1: Linear Probe for {model_name}\n{'='*50}") 
    
    model = initialize_model(model_name)
    report_efficiency(model) 
    
    # Setup for linear probe
    model, optimizer = setup_linear_probe(model) 
    
    # Train (max 30 epochs for full data)
    print("Training Linear Probe...")
    trained_model, history = train_model(model, train_loader, val_loader, optimizer, num_epochs=30) 
    
    # Save the weights for later visualization (PCA/t-SNE) 
    torch.save(trained_model.state_dict(), f"{model_name}_linear_probe.pth")
    return trained_model, history

In [ ]:
def main():
    for model_name in MODELS_TO_TEST:
        # 1. Linear Probe
        trained_model, history = run_linear_probe(model_name)
        
        # Get baseline clean accuracy for robustness testing
        clean_val_acc = history['val_acc'][-1]
        
if __name__ == "__main__":
    main()